# Classification using BoW with logistic regression, random forest and support vector machine 

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, balanced_accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import Normalizer
import os
from scipy.sparse import vstack

Load data

In [ ]:
# Load data
df = pd.read_csv('../../data/unfiltered/concatenated/preprocessed/author_documents_full_lemmatized.csv')
# unfiltered or filtered, stem or lemma
filename = "unfiltered_lem"
# Representation folder
path = "BoW"
# Inside representation folder
folder = "new"
print(f"Dataset loaded: {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
print(f"\nLabel distribution:")
print(df['label'].value_counts().sort_index())

# Labels: 
# 1 = Endometriosis (positive class)
# 2 = IBS (negative class)

df['target'] = (df['label'] == 1).astype(int)

print(f"\nBinary target distribution:")
target_counts = df['target'].value_counts().sort_index()
for target, count in target_counts.items():
    condition = "Endometriosis" if target == 1 else "IBS"
    percentage = (count / len(df)) * 100
    print(f"{condition}: {count:,} authors ({percentage:.1f}%)")

Initialise BoW representation

In [ ]:
X = df['concatenated_text']
y = df['target']

# Split into representation set and cross-validation set
X_rep, X_cv, y_rep, y_cv = train_test_split(
    X, y, test_size=0.8, random_state=42, stratify=y
)

# Example of X_rep
print(X_rep.head())

# Fit vectorizer on the representation set
vectorizer = CountVectorizer(
    max_features=5000, min_df=2, max_df=0.95,
    stop_words='english', ngram_range=(1, 2),
    token_pattern=r'\b[a-zA-Z]{2,}\b'
)
vectorizer.fit(X_rep)

# Transform everything using this fixed vectorizer
X_rep_bow = vectorizer.transform(X_rep)
X_cv_bow  = vectorizer.transform(X_cv)

# Normalize the vectors
normalizer = Normalizer(norm='l1')
X_rep_bow = normalizer.fit_transform(X_rep_bow)
X_cv_bow = normalizer.transform(X_cv_bow)


In [ ]:
print("Rep set text lengths (words):")
print(X_rep.str.split().str.len().describe())
print()
print("CV set text lengths (words):")
print(X_cv.str.split().str.len().describe())

Training logistic regression (LR), random forest (RF) and support vector machine (SVM)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42, C=1.0, gamma='scale', class_weight='balanced')
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")

    fold_f1_scores = []
    fold_details   = []

    oof_proba  = np.zeros(len(y_cv))   # one probability per CV sample
    oof_y_true = np.zeros(len(y_cv))   # matching true labels

    # k-fold CV on the 80%, adding rep set to training each time
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    fold_f1_scores, fold_details = [], []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_cv_bow, y_cv), 1):

        # k-1 folds from the CV set
        X_fold_train = X_cv_bow[train_idx]
        X_fold_val   = X_cv_bow[val_idx]
        y_fold_train = y_cv.iloc[train_idx]
        y_fold_val   = y_cv.iloc[val_idx]

        # Add the representation set to training data
        X_fold_train_full = vstack([X_rep_bow, X_fold_train])
        y_fold_train_full = pd.concat([y_rep, y_fold_train])

        # Train and evaluate
        fold_model = type(model)(**model.get_params())
        fold_model.fit(X_fold_train_full, y_fold_train_full)

        y_fold_pred = fold_model.predict(X_fold_val)
        y_fold_proba = fold_model.predict_proba(X_fold_val)[:, 1]

        oof_proba[val_idx]  = y_fold_proba
        oof_y_true[val_idx] = y_fold_val.values

        fold_f1_macro     = f1_score(y_fold_val, y_fold_pred, average='macro')
        fold_balanced_acc = balanced_accuracy_score(y_fold_val, y_fold_pred)
        f1_ibs  = f1_score(y_fold_val, y_fold_pred, pos_label=0)
        f1_endo = f1_score(y_fold_val, y_fold_pred, pos_label=1)
        fold_auc_roc = roc_auc_score(y_fold_val, fold_model.predict_proba(X_fold_val)[:, 1])

        fold_f1_scores.append(fold_f1_macro)
        fold_details.append({
            'fold':             fold,
            'macro_f1':         fold_f1_macro,
            'balanced_accuracy': fold_balanced_acc,
            'f1_ibs':           f1_ibs,
            'f1_endo':          f1_endo,
            'auc_roc':          fold_auc_roc
        })

        print(f"Fold {fold}: Macro F1 = {fold_f1_macro:.4f}, "
              f"Balanced Acc = {fold_balanced_acc:.4f} "
              f"(AUC-ROC  {fold_auc_roc:.4f})")
        
    cv_f1_mean  = np.mean(fold_f1_scores)
    cv_f1_std   = np.std(fold_f1_scores)

    cv_balanced_acc_scores = [d['balanced_accuracy'] for d in fold_details]
    cv_balanced_acc_mean   = np.mean(cv_balanced_acc_scores)
    cv_balanced_acc_std    = np.std(cv_balanced_acc_scores)
    
    cv_auc_roc_scores = [d['auc_roc'] for d in fold_details]
    cv_auc_roc = np.mean(cv_auc_roc_scores)

    results[name] = {
        'model':                  model,
        'cv_f1_mean':             cv_f1_mean,
        'cv_f1_std':              cv_f1_std,
        'cv_f1_scores':           fold_f1_scores,
        'cv_balanced_acc_mean':   cv_balanced_acc_mean,
        'cv_balanced_acc_std':    cv_balanced_acc_std,
        'cv_balanced_acc_scores': cv_balanced_acc_scores,
        'cv_f1_details':          fold_details,
        'cv_auc_roc_scores':      cv_auc_roc_scores,
        'auc-roc':                cv_auc_roc,
        'probabilities':          oof_proba,
        'true_labels':            oof_y_true
    }
    print(f"\n  10-Fold Cross Validation:")
    print(f"    CV Macro F1:         {cv_f1_mean:.4f} ± {cv_f1_std:.4f} "
          f"(range: {min(fold_f1_scores):.4f}–{max(fold_f1_scores):.4f})")
    print(f"    CV Balanced Accuracy: {cv_balanced_acc_mean:.4f} ± {cv_balanced_acc_std:.4f} "
          f"(range: {min(cv_balanced_acc_scores):.4f}–{max(cv_balanced_acc_scores):.4f})")
    print(f"    CV AUC-ROC: {cv_auc_roc:.4f}")

Save results

In [ ]:
# Save fold scores 
rows = []
for name, result in results.items():
    for fold_idx, (f1, bal_acc, auc_roc) in enumerate(
            zip(result['cv_f1_scores'], result['cv_balanced_acc_scores'], result['cv_auc_roc_scores']), 1):
        rows.append({
            'model':    name,
            'fold':     fold_idx,
            'CV mF1':   f1,
            'CV bACC':  bal_acc,
            'CV AUC-ROC': auc_roc
        })

fold_df = pd.DataFrame(rows)
fold_df.to_csv(
    f'../../data/output/{path}/{folder}/{filename}_foldscores.csv', index=False)


In [ ]:
all_model_results = []

for model_name, result in results.items():
    all_model_results.append({
        'Model':                    model_name,
        'CV Macro F1':              result['cv_f1_mean'],
        'CV Macro F1 Std':          result['cv_f1_std'],
        'CV Balanced Accuracy':     result['cv_balanced_acc_mean'],
        'CV Balanced Accuracy Std': result['cv_balanced_acc_std'],
        'CV AUC-ROC':               result['auc-roc'],
        'CV AUC-ROC Std':           np.std(result['cv_auc_roc_scores'])
    })

results_df = pd.DataFrame(all_model_results)
results_df.set_index('Model').T.to_csv(
    f'../../data/output/{path}/{folder}/{filename}_EM.csv')
print(results_df.to_string())

In [ ]:
# Save OOF probabilities for ROC curve plotting
roc_dir = f'../../data/output/{path}/{folder}/roc_values'
os.makedirs(roc_dir, exist_ok=True)

for name, result in results.items():
    safe_name = name.replace(' ', '_').lower()
    pd.DataFrame({
        'y_true':  result['true_labels'],
        'y_proba': result['probabilities']
    }).to_csv(
        os.path.join(roc_dir, f'{filename}_roc_{safe_name}.csv'), index=False)